##Notebook-2.5 (optional) - Testing different models for vector embedding

To find the most suitable model for the embedding and clustering of my crops, in order to discover the recurring character prints contained in them, I am testing three models on a curated set of "Editorial Cartoon" crops. The folder includes some of my character duplicates as well as noise. There are exactly 500 crops in the test-folder, as well as an xlsx-table, grouping the crops into with their respective duplicates. Groups 1 - 45 contain the duplicates, while group 0 defines noise.

###The three used models for testing:
- openai/clip-vit-base-patch32
- laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K
- Network with ResNet50-Architecture

##Environment
This notebook was created with the help of ChatGPT-5.5 and is supposed to be used in a Google Colab/Google Drive environment.



##1) Installing Dependencies and Importing Libraries

In [ ]:
#=================
# Installations
#=================
!pip -q install transformers==4.44.2 open_clip_torch==2.26.1 timm==1.0.9 \
                pillow tqdm matplotlib pandas scikit-learn
!pip -q install hdbscan || true

#showing the download versions
!pip show open_clip_torch
!pip show transformers

#========================
#Importing the libraries
#========================
import os, glob
import pandas as pd
import numpy as np
from pathlib import Path
import time
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import CLIPProcessor, CLIPModel
import open_clip
import torchvision
import torchvision.transforms as T
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    average_precision_score,
    roc_auc_score,
)
from sklearn.metrics import average_precision_score, roc_auc_score

##2) Mounting Google Drive + Setting Configurations

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Input
CARTOON_DIR = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Tests/crops_ed-cartoons_test"

# Expected columns: crops, group-id, numeric-order
GROUP_TABLE_PATH = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Tests/crops_ed-cartoons_test/test_grouping-table.xlsx"

# Output
OUT_DIR = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Tests/embedding_eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# If True, recompute embeddings even if cached .npy files exist in OUT_DIR
FORCE_RECOMPUTE = False

paths_cache = os.path.join(OUT_DIR, "paths.npy")

if (not FORCE_RECOMPUTE) and os.path.exists(paths_cache):
    paths = np.load(paths_cache, allow_pickle=True).tolist()
    print(f"Loaded cached paths from: {paths_cache}")
else:
    exts = ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp", "*.webp")
    paths = []
    for e in exts:
        paths.extend(glob.glob(os.path.join(CARTOON_DIR, "**", e), recursive=True))

    # deterministic + deduplicate
    paths = sorted(set(paths))

    np.save(paths_cache, np.array(paths, dtype=object))
    print(f"Saved paths to: {paths_cache}")

print("CARTOON_DIR:", CARTOON_DIR)
print("Number of images found:", len(paths))
print("First 5 paths:", paths[:5])

# --------------------------------------------------
# Load cleaned Excel table with group IDs
# --------------------------------------------------

group_df = pd.read_excel(GROUP_TABLE_PATH)

# - crops: image filename
# - group-id: manually assigned group number
required_columns = {"crops", "group-id"}
missing_columns = required_columns - set(group_df.columns)
if missing_columns:
    raise ValueError(
        f"Missing column(s) in group table: {missing_columns}. "
        f"Found columns: {list(group_df.columns)}"
    )

group_df = group_df[["crops", "group-id"]].copy()
group_df.columns = ["filename", "group_id"]

# Remove empty rows and clean values
group_df = group_df.dropna(subset=["filename", "group_id"])
group_df["filename"] = group_df["filename"].astype(str).str.strip()
group_df["group_id"] = pd.to_numeric(group_df["group_id"], errors="coerce")
group_df = group_df.dropna(subset=["group_id"])
group_df["group_id"] = group_df["group_id"].astype(int)

# Mapping: image filename -> manual group ID
filename_to_group = dict(zip(group_df["filename"], group_df["group_id"]))

# Align group IDs to the exact image path order used for the embeddings
path_filenames = [Path(p).name for p in paths]

gold_groups = []
missing_group_labels = []
for fname in path_filenames:
    if fname in filename_to_group:
        gold_groups.append(filename_to_group[fname])
    else:
        gold_groups.append(-999)  # internal marker for missing labels
        missing_group_labels.append(fname)

gold_groups = np.array(gold_groups, dtype=int)
valid_mask = gold_groups != -999

gold_eval = gold_groups[valid_mask]
paths_eval = np.array(paths, dtype=object)[valid_mask].tolist()

print("Images in embedding folder:", len(paths))
print("Images with group labels:", int(valid_mask.sum()))
print("Images missing group labels:", len(missing_group_labels))
print("Available manual groups:", sorted(set(gold_eval.tolist())))

if missing_group_labels:
    print("\nFirst missing filenames:")
    print(missing_group_labels[:20])

##3) Setting up the Test-Models + Creating Test-Embeddings

- Model A: CLIP-vit-base-patch32
- Model B: CLIP-ViT-L-14
- Model C: ResNet50-Architecture

In [ ]:
# shared embedding utilities
def load_image_rgb(path):
    img = Image.open(path).convert("RGB")
    return img


class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *args):
        self.t1 = time.perf_counter()
        self.dt = self.t1 - self.t0


def l2_normalize(x, eps=1e-12):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


def load_or_compute_embeddings(
    cache_path,
    embed_function,
    image_paths,
    batch_size,
):
    if (not FORCE_RECOMPUTE) and os.path.exists(cache_path):
        embeddings = np.load(cache_path).astype("float32")
        elapsed_time = float("nan")
        print(f"Loaded cached embeddings from: {cache_path}")
    else:
        embeddings, elapsed_time = embed_function(
            image_paths,
            batch_size=batch_size,
        )
        np.save(cache_path, embeddings)
        print(f"Saved embeddings to: {cache_path}")

    assert np.isfinite(embeddings).all(), \
        "Embeddings contain NaN/Inf."

    speed = (
        len(embeddings) / elapsed_time
        if np.isfinite(elapsed_time) and elapsed_time > 0
        else "n/a"
    )

    print(
        "Shape:", embeddings.shape,
        "| embed time (sec):", elapsed_time,
        "| imgs/sec:", speed,
    )

    return embeddings, elapsed_time

In [ ]:
# =====================================================
# Loading models and computing or loading embeddings
# =====================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding device: {device}")


# =====================================================
# MODEL A: Hugging Face CLIP ViT-B/32
# =====================================================
clipA_name = "openai/clip-vit-base-patch32"

clipA_model = (
    CLIPModel
    .from_pretrained(clipA_name)
    .to(device)
    .eval()
)

clipA_proc = CLIPProcessor.from_pretrained(clipA_name)


@torch.no_grad()
def embed_clip_hf(image_paths, batch_size=32):
    feats = []

    with Timer() as t:
        for i in tqdm(
            range(0, len(image_paths), batch_size),
            desc="CLIP-HF embedding",
        ):
            batch_paths = image_paths[i:i + batch_size]
            images = [load_image_rgb(path) for path in batch_paths]

            inputs = clipA_proc(
                images=images,
                return_tensors="pt",
            ).to(device)

            # Projected CLIP image features
            f = clipA_model.get_image_features(**inputs)
            f = f.detach().cpu().numpy().astype("float32")
            feats.append(f)

    feats = np.vstack(feats)
    feats = l2_normalize(feats)

    return feats, t.dt


embA, timeA = load_or_compute_embeddings(
    cache_path=os.path.join(
        OUT_DIR,
        "emb_clip_openai_vitb32.npy",
    ),
    embed_function=embed_clip_hf,
    image_paths=paths,
    batch_size=32,
)


# =====================================================
# MODEL B: OpenCLIP ViT-L/14
# =====================================================
clipB_model, _, clipB_preprocess = (
    open_clip.create_model_and_transforms(
        model_name="ViT-L-14",
        pretrained="datacomp_xl_s13b_b90k",
        device=device,
    )
)

clipB_model.eval()


@torch.no_grad()
def embed_openclip(image_paths, batch_size=16):
    feats = []

    with Timer() as t:
        for i in tqdm(
            range(0, len(image_paths), batch_size),
            desc="OpenCLIP embedding",
        ):
            batch_paths = image_paths[i:i + batch_size]

            images = [
                clipB_preprocess(load_image_rgb(path))
                for path in batch_paths
            ]

            x = torch.stack(images).to(device)

            f = clipB_model.encode_image(x)
            f = f.detach().cpu().numpy().astype("float32")
            feats.append(f)

    feats = np.vstack(feats)
    feats = l2_normalize(feats)

    return feats, t.dt


embB, timeB = load_or_compute_embeddings(
    cache_path=os.path.join(
        OUT_DIR,
        "emb_openclip_vitl14_datacomp_xl_s13b_b90k.npy",
    ),
    embed_function=embed_openclip,
    image_paths=paths,
    batch_size=16,
)


# =====================================================
# MODEL C: ResNet50
# =====================================================
resnet = torchvision.models.resnet50(
    weights=torchvision.models.ResNet50_Weights.DEFAULT
)

resnet.eval().to(device)

# Remove the final classification layer.
# Output shape after pooling: (batch_size, 2048, 1, 1)
resnet_feat = torch.nn.Sequential(
    *list(resnet.children())[:-1]
).to(device).eval()

resnet_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


@torch.no_grad()
def embed_resnet50(image_paths, batch_size=64):
    feats = []

    with Timer() as t:
        for i in tqdm(
            range(0, len(image_paths), batch_size),
            desc="ResNet50 embedding",
        ):
            batch_paths = image_paths[i:i + batch_size]

            images = [
                resnet_tf(load_image_rgb(path))
                for path in batch_paths
            ]

            x = torch.stack(images).to(device)

            f = resnet_feat(x)             # (B, 2048, 1, 1)
            f = f.squeeze(-1).squeeze(-1)  # (B, 2048)
            f = f.detach().cpu().numpy().astype("float32")
            feats.append(f)

    feats = np.vstack(feats)
    feats = l2_normalize(feats)

    return feats, t.dt


embC, timeC = load_or_compute_embeddings(
    cache_path=os.path.join(
        OUT_DIR,
        "emb_resnet50.npy",
    ),
    embed_function=embed_resnet50,
    image_paths=paths,
    batch_size=64,
)

In [ ]:
# =========================================================================
# Evaluation embeddings: keeping only images that have manual group labels
# =========================================================================
embA_eval = embA[valid_mask]
embB_eval = embB[valid_mask]
embC_eval = embC[valid_mask]

print("Evaluation subset sizes:")
print("  CLIP-B/32:", embA_eval.shape)
print("  CLIP-L/14 DataComp:", embB_eval.shape)
print("  ResNet50:", embC_eval.shape)
print("  gold_eval:", gold_eval.shape)

assert len(embA_eval) == len(gold_eval) == len(paths_eval)
assert len(embB_eval) == len(gold_eval)
assert len(embC_eval) == len(gold_eval)


##4) Retrieval and clustering evaluation


In [ ]:
# =====================================================
# Auxiliary functions
# =====================================================

# =====================================================
# 1. Retrieval evaluation
# =====================================================
def evaluate_retrieval_by_group(
    emb,
    gold_groups,
    model_name,
    ks=(1, 3, 5, 10),
    ignore_group_zero=True,
):
    """
    For every labelled query image, rank all other images by cosine
    similarity.

    A retrieval is correct if the retrieved image has the same manual
    group ID as the query image.

    By default, group ID 0 is treated as an individual/noise image and
    excluded as a query.
    """
    emb = np.asarray(emb, dtype="float32")
    gold_groups = np.asarray(gold_groups)

    similarity_matrix = cosine_similarity(emb)

    # Exclude the query image itself from its own ranking.
    np.fill_diagonal(similarity_matrix, -np.inf)

    valid_query_indices = []

    for i, group_id in enumerate(gold_groups):
        if ignore_group_zero and group_id == 0:
            continue

        # Evaluate only groups that contain at least one matching partner.
        if np.sum(gold_groups == group_id) > 1:
            valid_query_indices.append(i)

    rows = []

    for k in ks:
        precisions = []
        recalls = []

        for i in valid_query_indices:
            group_id = gold_groups[i]

            ranking = np.argsort(similarity_matrix[i])[::-1]
            top_k = ranking[:k]

            correct = np.sum(gold_groups[top_k] == group_id)
            possible_correct = np.sum(gold_groups == group_id) - 1

            precisions.append(correct / k)
            recalls.append(
                correct / possible_correct
                if possible_correct > 0
                else 0
            )

        rows.append({
            "model": model_name,
            "k": k,
            "precision@k": (
                float(np.mean(precisions))
                if precisions
                else np.nan
            ),
            "recall@k": (
                float(np.mean(recalls))
                if recalls
                else np.nan
            ),
            "evaluated_queries": len(valid_query_indices),
        })

    return pd.DataFrame(rows)


def evaluate_pairwise_similarity(
    emb,
    gold_groups,
    model_name,
    ignore_group_zero=True,
):
    """
    Build all possible image pairs and test whether same-group pairs
    receive higher cosine similarity scores than different-group pairs.
    """
    emb = np.asarray(emb, dtype="float32")
    gold_groups = np.asarray(gold_groups)

    similarity_matrix = cosine_similarity(emb)

    y_true = []
    y_score = []

    n_images = len(gold_groups)

    for i in range(n_images):
        for j in range(i + 1, n_images):
            if (
                ignore_group_zero
                and (
                    gold_groups[i] == 0
                    or gold_groups[j] == 0
                )
            ):
                continue

            y_true.append(
                int(gold_groups[i] == gold_groups[j])
            )
            y_score.append(
                float(similarity_matrix[i, j])
            )

    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # These metrics require at least one positive and one negative pair.
    if len(set(y_true.tolist())) < 2:
        average_precision = np.nan
        roc_auc = np.nan
    else:
        average_precision = average_precision_score(
            y_true,
            y_score,
        )
        roc_auc = roc_auc_score(
            y_true,
            y_score,
        )

    return {
        "model": model_name,
        "average_precision": average_precision,
        "roc_auc": roc_auc,
        "positive_pairs": int(np.sum(y_true == 1)),
        "negative_pairs": int(np.sum(y_true == 0)),
    }


# =====================================================
# 2. Creating Singletons from Group 0
# =====================================================

def make_noise_singletons(
    gold_groups,
    noise_label=0,
):
    """
    Replace every occurrence of the ground-truth noise label with a
    unique singleton label.

    Example
    -------
    [0, 0, 1, 1, 2, 0]
        ↓
    [3, 4, 1, 1, 2, 5]

    This allows individual/noise images to remain in the evaluation
    without being treated as one common cluster.
    """
    gold_groups = np.asarray(gold_groups).copy()

    noise_indices = np.where(gold_groups == noise_label)[0]

    if len(noise_indices) == 0:
        return gold_groups

    next_label = int(gold_groups.max()) + 1

    for i, idx in enumerate(noise_indices):
        gold_groups[idx] = next_label + i

    return gold_groups


# =====================================================
# 3. Nearest-neighbour search and visualization
# =====================================================
def build_knn_cosine_index(emb):
    """
    Build a nearest-neighbour index using cosine distance.
    """
    emb = np.asarray(emb, dtype="float32")

    nearest_neighbors = NearestNeighbors(
        metric="cosine",
        algorithm="brute",
    )

    nearest_neighbors.fit(emb)

    return nearest_neighbors


def knn_search(nearest_neighbors, queries, k=10):
    """
    Retrieve the k nearest neighbours and convert cosine distances
    into cosine similarities.
    """
    queries = np.asarray(queries, dtype="float32")

    distances, indices = nearest_neighbors.kneighbors(
        queries,
        n_neighbors=k,
        return_distance=True,
    )

    similarities = 1.0 - distances

    return similarities, indices


def benchmark_knn(
    nearest_neighbors,
    emb,
    n_queries=200,
    k=10,
    seed=0,
):
    """
    Run k-nearest-neighbour searches for a random sample of embeddings
    and measure the total search time.
    """
    emb = np.asarray(emb, dtype="float32")

    rng = np.random.default_rng(seed)

    query_indices = rng.choice(
        len(emb),
        size=min(n_queries, len(emb)),
        replace=False,
    )

    queries = emb[query_indices]

    start_time = time.perf_counter()

    similarities, neighbour_indices = knn_search(
        nearest_neighbors,
        queries,
        k=k,
    )

    elapsed_time = time.perf_counter() - start_time

    return (
        elapsed_time,
        len(query_indices),
        similarities,
        neighbour_indices,
        query_indices,
    )


def show_retrieval_grid(
    paths,
    similarities,
    neighbour_indices,
    k=10,
    title="",
):
    """
    Display one query image and its nearest neighbours.

    This function assumes that the query image itself is returned at
    rank 0.
    """
    cols = 6
    rows = int(np.ceil((k + 1) / cols))

    plt.figure(figsize=(14, 6))

    for rank in range(k + 1):
        ax = plt.subplot(rows, cols, rank + 1)

        image_path = paths[neighbour_indices[rank]]
        image = Image.open(image_path).convert("RGB")

        ax.imshow(image)

        if rank == 0:
            ax.set_title("QUERY")
        else:
            ax.set_title(
                f"{rank}: {similarities[rank]:.3f}"
            )

        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def inspect_model_knn(
    emb,
    paths,
    model_name,
    query_indices=None,
    k=10,
):
    """
    Display nearest-neighbour retrieval results for selected queries.
    """
    nearest_neighbors = build_knn_cosine_index(emb)

    if query_indices is None:
        query_indices = [0, 1, 2, 3, 4]

    for query_index in query_indices:
        query = emb[query_index:query_index + 1]

        similarities, neighbour_indices = knn_search(
            nearest_neighbors,
            query,
            k=k + 1,
        )

        show_retrieval_grid(
            paths=paths,
            similarities=similarities[0],
            neighbour_indices=neighbour_indices[0],
            k=k,
            title=(
                f"{model_name} | "
                f"query={query_index} | "
                f"{Path(paths[query_index]).name}"
            ),
        )


# ===================================================================================
# 4. Clustering algorithms: DBSCAN, HDBSCAN, Threshold Graphing and Complete Linkage
# ===================================================================================
def cluster_dbscan_cosine(
    emb,
    eps=0.08,
    min_samples=2,
):
    """
    Run DBSCAN using a precomputed cosine-distance matrix.
    """
    emb = np.asarray(emb, dtype="float32")

    distance_matrix = cosine_distances(emb)

    clusterer = DBSCAN(
        eps=eps,
        min_samples=min_samples,
        metric="precomputed",
    )

    labels = clusterer.fit_predict(distance_matrix)

    return labels


def sweep_dbscan(
    emb,
    eps_values,
    min_samples=2,
):
    """
    Run DBSCAN for several epsilon values and return a summary table.
    """
    rows = []

    for eps in eps_values:
        labels = cluster_dbscan_cosine(
            emb,
            eps=eps,
            min_samples=min_samples,
        )

        summary = summarize_clusters(labels)

        rows.append({
            "eps": eps,
            "clusters": summary["n_clusters"],
            "noise": summary["noise"],
        })

    return pd.DataFrame(rows)


def cluster_hdbscan(
    emb,
    min_cluster_size=3,
):
    """
    Run HDBSCAN if the optional hdbscan package is available.
    """
    emb = np.asarray(emb, dtype="float32")

    try:
        import hdbscan

        clusterer = hdbscan.HDBSCAN(
            metric="euclidean",
            min_cluster_size=min_cluster_size,
        )

        labels = clusterer.fit_predict(emb)

        return labels

    except Exception as error:
        print(
            "HDBSCAN not available or failed:",
            error,
        )

        return None


def threshold_clusters(
    emb,
    sim_thresh=0.90,
):
    """
    Construct clusters as connected components in a similarity graph.

    Two images are connected when their cosine similarity is greater
    than or equal to sim_thresh. Indirect connections can therefore
    place images in the same cluster even when not every image pair
    reaches the threshold.
    """
    emb = np.asarray(emb, dtype="float32")

    similarity_matrix = cosine_similarity(emb)

    np.fill_diagonal(similarity_matrix, 0.0)

    n_images = similarity_matrix.shape[0]

    visited = np.zeros(
        n_images,
        dtype=bool,
    )

    labels = -np.ones(
        n_images,
        dtype=int,
    )

    cluster_id = 0

    for i in range(n_images):
        if visited[i]:
            continue

        stack = [i]
        visited[i] = True
        component = [i]

        while stack:
            current = stack.pop()

            neighbours = np.where(
                similarity_matrix[current] >= sim_thresh
            )[0]

            for neighbour in neighbours:
                if not visited[neighbour]:
                    visited[neighbour] = True
                    stack.append(neighbour)
                    component.append(neighbour)

        # Keep only clusters with at least two images.
        # Single images remain labelled as noise (-1).
        if len(component) >= 2:
            for image_index in component:
                labels[image_index] = cluster_id

            cluster_id += 1

    return labels


def greedy_complete_link_clusters(
    emb,
    sim_thresh=0.90,
    min_size=2,
):
    """
    Greedily construct clusters in which every pair of images reaches
    the specified cosine-similarity threshold.

    This is a custom greedy threshold procedure, not standard
    hierarchical agglomerative complete-link clustering.

    For every cluster C, the intended condition is:

        cosine_similarity(i, j) >= sim_thresh

    for every pair i, j in C.
    """
    emb = np.asarray(emb, dtype="float32")

    similarity_matrix = cosine_similarity(emb)

    np.fill_diagonal(
        similarity_matrix,
        1.0,
    )

    adjacency = similarity_matrix >= sim_thresh

    n_images = adjacency.shape[0]

    labels = -np.ones(
        n_images,
        dtype=int,
    )

    unassigned = set(range(n_images))
    cluster_id = 0

    neighbour_sets = [
        set(np.where(adjacency[i])[0].tolist())
        for i in range(n_images)
    ]

    while unassigned:
        # Select the unassigned image with the most currently
        # available neighbours.
        seed = max(
            unassigned,
            key=lambda i: len(
                neighbour_sets[i] & unassigned
            ),
        )

        cluster = {seed}

        candidates = (
            neighbour_sets[seed]
            & unassigned
        ) - {seed}

        while candidates:
            # Prefer the candidate with the highest degree among
            # the remaining candidates.
            candidate = max(
                candidates,
                key=lambda i: len(
                    neighbour_sets[i] & candidates
                ),
            )

            cluster.add(candidate)

            # A remaining candidate must be connected to every
            # image already included in the cluster.
            candidates = (
                candidates
                & neighbour_sets[candidate]
                & unassigned
            ) - cluster

        if len(cluster) >= min_size:
            for image_index in cluster:
                labels[image_index] = cluster_id

            cluster_id += 1

        unassigned -= cluster

    return labels


# =====================================================
# 5. Cluster summaries and evaluation
# =====================================================
def summarize_clusters(labels):
    """
    Return the number of clusters, the number of noise images, and
    the sizes of the ten largest clusters.
    """
    labels = np.asarray(labels)

    cluster_ids = [
        cluster_id
        for cluster_id in sorted(set(labels))
        if cluster_id != -1
    ]

    cluster_sizes = sorted(
        [
            (
                cluster_id,
                int((labels == cluster_id).sum()),
            )
            for cluster_id in cluster_ids
        ],
        key=lambda item: item[1],
        reverse=True,
    )

    return {
        "N": int(len(labels)),
        "n_clusters": int(len(cluster_ids)),
        "noise": int((labels == -1).sum()),
        "top_sizes": cluster_sizes[:10],
    }


def _noise_as_singletons(labels):
    """
    Convert every noise assignment (-1) into a unique singleton label.

    ARI and NMI otherwise interpret all -1 values as one shared cluster,
    even though clustering noise means that those images are unassigned.
    """
    labels = np.asarray(labels, dtype=int).copy()

    noise_indices = np.where(labels == -1)[0]

    if len(noise_indices) == 0:
        return labels

    next_label = (
        int(labels[labels != -1].max()) + 1
        if np.any(labels != -1)
        else 0
    )

    labels[noise_indices] = np.arange(
        next_label,
        next_label + len(noise_indices),
    )

    return labels


def evaluate_clusters_against_groups(
    labels,
    gold_groups,
    model_name,
    algorithm=None,
    parameter=None,
    ignore_group_zero=False,
):
    """
    Compare one predicted clustering with the manual groups.

    Ground-truth group 0 denotes unrelated individual images.

    If ignore_group_zero is False, every group-0 image is treated as a
    separate singleton in the evaluation.

    If ignore_group_zero is True, group-0 images are excluded.
    """
    labels = np.asarray(labels)
    gold_groups_original = np.asarray(gold_groups)

    if len(labels) != len(gold_groups_original):
        raise ValueError(
            "Predicted labels and gold groups must have equal length."
        )

    evaluation_mask = np.ones(
        len(gold_groups_original),
        dtype=bool,
    )

    if ignore_group_zero:
        evaluation_mask &= gold_groups_original != 0

    labels_eval = labels[evaluation_mask]
    gold_eval_local = gold_groups_original[evaluation_mask]

    # When group 0 is included, treat each of its images as a separate
    # singleton rather than as one shared manual cluster.
    if not ignore_group_zero:
        gold_eval_local = make_noise_singletons(
            gold_eval_local,
            noise_label=0,
        )

    if len(gold_eval_local) < 2:
        raise ValueError(
            "At least two evaluated images are required."
        )

    # Pairwise evaluation on the upper triangle of the pair matrix.
    upper_i, upper_j = np.triu_indices(
        len(gold_eval_local),
        k=1,
    )

    true_same = (
        gold_eval_local[upper_i]
        == gold_eval_local[upper_j]
    )

    # Predicted noise images do not form a shared predicted cluster.
    predicted_same = (
        (labels_eval[upper_i] == labels_eval[upper_j])
        & (labels_eval[upper_i] != -1)
    )

    true_positives = int(
        np.sum(true_same & predicted_same)
    )
    false_positives = int(
        np.sum(~true_same & predicted_same)
    )
    false_negatives = int(
        np.sum(true_same & ~predicted_same)
    )

    pairwise_precision = (
        true_positives / (true_positives + false_positives)
        if (true_positives + false_positives) > 0
        else 0.0
    )

    pairwise_recall = (
        true_positives / (true_positives + false_negatives)
        if (true_positives + false_negatives) > 0
        else 0.0
    )

    pairwise_f1 = (
        2 * pairwise_precision * pairwise_recall
        / (pairwise_precision + pairwise_recall)
        if (pairwise_precision + pairwise_recall) > 0
        else 0.0
    )

    # ARI and NMI also need every predicted noise image to be represented
    # as its own singleton rather than as one shared -1 cluster.
    labels_for_partition_metrics = _noise_as_singletons(
        labels_eval
    )

    non_noise_cluster_ids = (
        set(labels_eval.tolist()) - {-1}
    )
    predicted_noise = int(
        np.sum(labels_eval == -1)
    )

    return {
        "model": model_name,
        "algorithm": algorithm,
        "parameter": parameter,
        "pairwise_precision": pairwise_precision,
        "pairwise_recall": pairwise_recall,
        "pairwise_F1": pairwise_f1,
        "ARI": adjusted_rand_score(
            gold_eval_local,
            labels_for_partition_metrics,
        ),
        "NMI": normalized_mutual_info_score(
            gold_eval_local,
            labels_for_partition_metrics,
        ),
        "evaluated_images": int(len(gold_eval_local)),
        "manual_groups": int(
            len(set(gold_eval_local.tolist()))
        ),
        "predicted_clusters": int(
            len(non_noise_cluster_ids)
        ),
        "predicted_noise": predicted_noise,
        "noise_fraction": (
            predicted_noise / len(labels_eval)
        ),
        "true_positive_pairs": true_positives,
        "false_positive_pairs": false_positives,
        "false_negative_pairs": false_negatives,
    }

def add_benchmark_result(
    model_name,
    algorithm,
    parameter,
    labels,
    gold_groups,
    benchmark_rows,
    benchmark_labels,
    ignore_group_zero=False,
):
    """
    Evaluate one clustering configuration against the manual ground
    truth and store both its metrics and predicted cluster labels.
    """
    result = evaluate_clusters_against_groups(
        labels=labels,
        gold_groups=gold_groups,
        model_name=model_name,
        algorithm=algorithm,
        parameter=parameter,
        ignore_group_zero=ignore_group_zero,
    )

    benchmark_rows.append(result)

    benchmark_labels[
        (model_name, algorithm, parameter)
    ] = np.asarray(labels).copy()


# =====================================================
# 6. Cluster and noise visualization
# =====================================================
def show_cluster(
    labels,
    cluster_id,
    paths,
    max_show=None,
    cols=6,
):
    """
    Display images belonging to one cluster.

    If max_show is None, all images in the cluster are displayed.
    """
    labels = np.asarray(labels)

    image_indices = np.where(
        labels == cluster_id
    )[0].tolist()

    if len(image_indices) == 0:
        print(
            "No items in cluster",
            cluster_id,
        )
        return

    if max_show is not None:
        image_indices = image_indices[:max_show]

    rows = int(
        np.ceil(len(image_indices) / cols)
    )

    plt.figure(
        figsize=(14, 2.6 * rows)
    )

    for plot_index, image_index in enumerate(image_indices):
        ax = plt.subplot(
            rows,
            cols,
            plot_index + 1,
        )

        image = Image.open(
            paths[image_index]
        ).convert("RGB")

        ax.imshow(image)
        ax.set_title(str(image_index))
        ax.axis("off")

    total_cluster_size = int(
        (labels == cluster_id).sum()
    )

    plt.suptitle(
        f"Cluster {cluster_id} | "
        f"size={total_cluster_size}"
    )

    plt.tight_layout()
    plt.show()


def show_cluster_paged(
    labels,
    cluster_id,
    paths,
    page_size=24,
    cols=6,
):
    """
    Display all images in a cluster across several figure pages.
    """
    labels = np.asarray(labels)

    image_indices = np.where(
        labels == cluster_id
    )[0].tolist()

    total = len(image_indices)

    if total == 0:
        print(
            "No items in cluster",
            cluster_id,
        )
        return

    print(
        f"Cluster {cluster_id} | "
        f"total items = {total}"
    )

    for start in range(0, total, page_size):
        end = min(
            start + page_size,
            total,
        )

        page_indices = image_indices[start:end]

        rows = int(
            np.ceil(len(page_indices) / cols)
        )

        plt.figure(
            figsize=(14, 2.6 * rows)
        )

        for plot_index, image_index in enumerate(page_indices):
            ax = plt.subplot(
                rows,
                cols,
                plot_index + 1,
            )

            image = Image.open(
                paths[image_index]
            ).convert("RGB")

            ax.imshow(image)
            ax.set_title(str(image_index))
            ax.axis("off")

        plt.suptitle(
            f"Cluster {cluster_id} | "
            f"showing {start + 1}-{end} of {total}"
        )

        plt.tight_layout()
        plt.show()


def show_top_clusters(
    labels,
    paths,
    top_n=5,
    max_show=None,
    title="",
    page_size=24,
    use_pagination=True,
):
    """
    Summarize and display the largest predicted clusters.
    """
    summary = summarize_clusters(labels)

    print(title, summary)

    for cluster_id, cluster_size in summary["top_sizes"][:top_n]:
        if (
            use_pagination
            and max_show is None
            and cluster_size > page_size
        ):
            show_cluster_paged(
                labels=labels,
                cluster_id=cluster_id,
                paths=paths,
                page_size=page_size,
            )
        else:
            show_cluster(
                labels=labels,
                cluster_id=cluster_id,
                paths=paths,
                max_show=max_show,
            )


def show_indices_grid(
    indices,
    paths,
    title="",
    max_show=60,
    cols=10,
):
    """
    Display selected image indices in a grid.
    """
    indices = list(indices)[:max_show]

    if len(indices) == 0:
        print(
            title,
            "- no items",
        )
        return

    rows = int(
        np.ceil(len(indices) / cols)
    )

    plt.figure(
        figsize=(2.0 * cols, 2.0 * rows)
    )

    for plot_index, image_index in enumerate(indices):
        ax = plt.subplot(
            rows,
            cols,
            plot_index + 1,
        )

        image = Image.open(
            paths[image_index]
        ).convert("RGB")

        ax.imshow(image)
        ax.set_title(
            str(image_index),
            fontsize=9,
        )
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_noise_for_model(
    labels,
    paths,
    model_name,
    max_show=80,
):
    """
    Display images that were assigned the noise label -1.
    """
    labels = np.asarray(labels)

    noise_indices = np.where(
        labels == -1
    )[0]

    print(
        f"{model_name} noise count:",
        len(noise_indices),
    )

    show_indices_grid(
        indices=noise_indices,
        paths=paths,
        title=(
            f"{model_name} — NOISE (label=-1), "
            f"showing first "
            f"{min(max_show, len(noise_indices))}"
        ),
        max_show=max_show,
        cols=10,
    )

    return noise_indices

### Step-by-step evaluation and model selection

The clustering benchmark below runs every method directly on the manually labelled evaluation subset. Each configuration is compared with the ground-truth character groups using pairwise precision, recall, F1, ARI, and NMI.


In [ ]:
# =====================================================
# 1. Retrieval evaluation
# =====================================================

retrieval_results = pd.concat(
    [
        evaluate_retrieval_by_group(
            embA_eval,
            gold_eval,
            "CLIP-B/32 HF",
        ),
        evaluate_retrieval_by_group(
            embB_eval,
            gold_eval,
            "CLIP-L/14 DataComp",
        ),
        evaluate_retrieval_by_group(
            embC_eval,
            gold_eval,
            "ResNet50",
        ),
    ],
    ignore_index=True,
)

display(retrieval_results)


pairwise_results = pd.DataFrame(
    [
        evaluate_pairwise_similarity(
            embA_eval,
            gold_eval,
            "CLIP-B/32 HF",
        ),
        evaluate_pairwise_similarity(
            embB_eval,
            gold_eval,
            "CLIP-L/14 DataComp",
        ),
        evaluate_pairwise_similarity(
            embC_eval,
            gold_eval,
            "ResNet50",
        ),
    ]
)

display(pairwise_results)

In [ ]:
# =====================================================
# 2. Nearest-neighbour benchmark and inspection
# =====================================================

model_data = [
    {
        "name": "CLIP-B/32 HF",
        "embeddings": embA,
        "embedding_time": timeA,
    },
    {
        "name": "CLIP-L/14 DataComp",
        "embeddings": embB,
        "embedding_time": timeB,
    },
    {
        "name": "ResNet50",
        "embeddings": embC,
        "embedding_time": timeC,
    },
]


# Benchmark KNN retrieval speed
for model in model_data:
    name = model["name"]
    embeddings = model["embeddings"]
    embedding_time = model["embedding_time"]

    knn_index = build_knn_cosine_index(embeddings)

    (
        query_time,
        n_queries,
        similarities,
        neighbour_indices,
        query_indices,
    ) = benchmark_knn(
        knn_index,
        embeddings,
        n_queries=200,
        k=10,
    )

    if np.isfinite(embedding_time) and embedding_time > 0:
        images_per_second = len(embeddings) / embedding_time
        embedding_time_text = f"{embedding_time:.3f} sec"
        speed_text = f"{images_per_second:.2f} images/sec"
    else:
        embedding_time_text = "n/a"
        speed_text = "n/a"

    milliseconds_per_query = (
        1000 * query_time / n_queries
    )

    print(f"\n{name}")
    print(
        f"  Embedding time: {embedding_time_text}"
        f" | {speed_text}"
    )
    print(
        f"  Query batch: {query_time:.4f} sec "
        f"for {n_queries} queries"
        f" | {milliseconds_per_query:.3f} ms/query"
    )


# Visually inspect selected retrieval examples
query_indices_to_show = [0, 10, 20]

for model in model_data:
    inspect_model_knn(
        emb=model["embeddings"],
        paths=paths,
        model_name=model["name"],
        query_indices=query_indices_to_show,
        k=10,
    )

In [ ]:
# =====================================================
# 3. Comprehensive clustering benchmark against GT
# =====================================================

clustering_models_eval = [
    ("CLIP-B/32 HF", embA_eval),
    ("CLIP-L/14 DataComp", embB_eval),
    ("ResNet50", embC_eval),
]

# Parameter grids to compare
HDBSCAN_MIN_CLUSTER_SIZES = [2, 3, 4, 5, 8, 10]
DBSCAN_EPS_VALUES = [
    0.03,
    0.05,
    0.07,
    0.08,
    0.10,
    0.12,
    0.15,
]
SIMILARITY_THRESHOLDS = [
    0.80,
    0.85,
    0.88,
    0.90,
    0.92,
    0.94,
    0.96,
]

benchmark_rows = []
benchmark_labels = {}


for model_name, embeddings in clustering_models_eval:
    print(f"Benchmarking {model_name} ...")

    # HDBSCAN sweep
    for min_cluster_size in HDBSCAN_MIN_CLUSTER_SIZES:
        labels = cluster_hdbscan(
            embeddings,
            min_cluster_size=min_cluster_size,
        )

        if labels is not None:
            add_benchmark_result(
                model_name=model_name,
                algorithm="HDBSCAN",
                parameter=f"min_cluster_size={min_cluster_size}",
                labels=labels,
                gold_groups=gold_eval,
                benchmark_rows=benchmark_rows,
                benchmark_labels=benchmark_labels,
                ignore_group_zero=False,
            )

    # DBSCAN sweep
    for eps in DBSCAN_EPS_VALUES:
        labels = cluster_dbscan_cosine(
            embeddings,
            eps=eps,
            min_samples=2,
        )

        # Debug output for the currently tested DBSCAN configuration
        print(
            f"\n===== {model_name} | DBSCAN "
            f"eps={eps:.2f}, min_samples=2 ====="
        )
        print("Unique predicted labels:", np.unique(labels))
        print("Noise images (-1):", np.sum(labels == -1))
        print("Cluster sizes:")
        print(pd.Series(labels).value_counts().sort_index())

        add_benchmark_result(
            model_name=model_name,
            algorithm="DBSCAN",
            parameter=f"eps={eps:.2f}, min_samples=2",
            labels=labels,
            gold_groups=gold_eval,
            benchmark_rows=benchmark_rows,
            benchmark_labels=benchmark_labels,
            ignore_group_zero=False,
        )

    # Connected-component similarity-threshold sweep
    for similarity_threshold in SIMILARITY_THRESHOLDS:
        labels = threshold_clusters(
            embeddings,
            sim_thresh=similarity_threshold,
        )

        add_benchmark_result(
            model_name=model_name,
            algorithm="Threshold connected components",
            parameter=f"similarity={similarity_threshold:.2f}",
            labels=labels,
            gold_groups=gold_eval,
            benchmark_rows=benchmark_rows,
            benchmark_labels=benchmark_labels,
            ignore_group_zero=False,
        )

    # Greedy complete-link similarity-threshold sweep
    for similarity_threshold in SIMILARITY_THRESHOLDS:
        labels = greedy_complete_link_clusters(
            embeddings,
            sim_thresh=similarity_threshold,
            min_size=2,
        )

        add_benchmark_result(
            model_name=model_name,
            algorithm="Greedy complete-link",
            parameter=f"similarity={similarity_threshold:.2f}",
            labels=labels,
            gold_groups=gold_eval,
            benchmark_rows=benchmark_rows,
            benchmark_labels=benchmark_labels,
            ignore_group_zero=False,
        )


# Rank primarily by pairwise F1, then use precision, ARI, NMI, and
# lower noise as tie-breakers.
clustering_benchmark_results = (
    pd.DataFrame(benchmark_rows)
    .sort_values(
        by=[
            "pairwise_F1",
            "pairwise_precision",
            "ARI",
            "NMI",
            "noise_fraction",
        ],
        ascending=[False, False, False, False, True],
    )
    .reset_index(drop=True)
)

clustering_benchmark_results.insert(
    0,
    "rank",
    np.arange(1, len(clustering_benchmark_results) + 1),
)

# Main ranked table.
display(
    clustering_benchmark_results[
        [
            "rank",
            "model",
            "algorithm",
            "parameter",
            "pairwise_F1",
            "pairwise_precision",
            "pairwise_recall",
            "ARI",
            "NMI",
            "predicted_clusters",
            "predicted_noise",
            "noise_fraction",
            "evaluated_images",
            "manual_groups",
        ]
    ].style.format({
        "pairwise_F1": "{:.4f}",
        "pairwise_precision": "{:.4f}",
        "pairwise_recall": "{:.4f}",
        "ARI": "{:.4f}",
        "NMI": "{:.4f}",
        "noise_fraction": "{:.2%}",
    })
)

# Compact comparison: best configuration for each model/algorithm pair.
best_per_model_algorithm = (
    clustering_benchmark_results
    .sort_values("rank")
    .groupby(
        ["model", "algorithm"],
        as_index=False,
        sort=False,
    )
    .first()
    .sort_values("rank")
    .reset_index(drop=True)
)

print("\nBest parameter setting for each model and algorithm:")
display(
    best_per_model_algorithm[
        [
            "rank",
            "model",
            "algorithm",
            "parameter",
            "pairwise_F1",
            "pairwise_precision",
            "pairwise_recall",
            "ARI",
            "NMI",
            "predicted_clusters",
            "predicted_noise",
        ]
    ].style.format({
        "pairwise_F1": "{:.4f}",
        "pairwise_precision": "{:.4f}",
        "pairwise_recall": "{:.4f}",
        "ARI": "{:.4f}",
        "NMI": "{:.4f}",
    })
)

CLUSTERING_RESULTS_CSV = os.path.join(
    OUT_DIR,
    "clustering_algorithm_GT_benchmark.csv",
)

clustering_benchmark_results.to_csv(
    CLUSTERING_RESULTS_CSV,
    index=False,
)

print(f"\nSaved benchmark to: {CLUSTERING_RESULTS_CSV}")


In [ ]:
# =====================================================
# 4. Automatic best-configuration summary
# =====================================================

best_configuration = clustering_benchmark_results.iloc[0]

BEST_MODEL = best_configuration["model"]
BEST_ALGORITHM = best_configuration["algorithm"]
BEST_PARAMETER = best_configuration["parameter"]

best_key = (
    BEST_MODEL,
    BEST_ALGORITHM,
    BEST_PARAMETER,
)

best_labels_eval = benchmark_labels[best_key]

# Path for the exported summary.
BEST_CONFIGURATION_TXT = os.path.join(
    OUT_DIR,
    "best_clustering_configuration.txt",
)

# Build the summary once so the same content can be printed and saved.
best_summary = "\n".join([
    "=" * 68,
    "BEST CLUSTERING CONFIGURATION ON THE MANUAL GROUND TRUTH",
    "=" * 68,
    f"Embedding model:       {BEST_MODEL}",
    f"Clustering algorithm:  {BEST_ALGORITHM}",
    f"Parameter setting:     {BEST_PARAMETER}",
    "-" * 68,
    (
        f"Pairwise precision:   "
        f"{best_configuration['pairwise_precision']:.4f}"
    ),
    (
        f"Pairwise recall:      "
        f"{best_configuration['pairwise_recall']:.4f}"
    ),
    (
        f"Pairwise F1:          "
        f"{best_configuration['pairwise_F1']:.4f}"
    ),
    f"ARI:                  {best_configuration['ARI']:.4f}",
    f"NMI:                  {best_configuration['NMI']:.4f}",
    (
        f"Predicted clusters:   "
        f"{int(best_configuration['predicted_clusters'])}"
    ),
    (
        f"Predicted noise:      "
        f"{int(best_configuration['predicted_noise'])} "
        f"({best_configuration['noise_fraction']:.2%})"
    ),
    (
        f"Evaluated images:     "
        f"{int(best_configuration['evaluated_images'])}"
    ),
    (
        f"Manual groups:        "
        f"{int(best_configuration['manual_groups'])}"
    ),
    "=" * 68,
])

# Print the summary in the notebook.
print(best_summary)

# Save the summary as a TXT file.
with open(
    BEST_CONFIGURATION_TXT,
    "w",
    encoding="utf-8",
) as file:
    file.write(best_summary)
    file.write("\n")

print(
    f"\nSaved best-configuration summary to:\n"
    f"{BEST_CONFIGURATION_TXT}"
)

# Show the ten strongest configurations for a quick robustness check.
print("\nTop 10 configurations:")

display(
    clustering_benchmark_results.head(10)[
        [
            "rank",
            "model",
            "algorithm",
            "parameter",
            "pairwise_F1",
            "pairwise_precision",
            "pairwise_recall",
            "ARI",
            "NMI",
            "predicted_clusters",
            "predicted_noise",
        ]
    ].style.format({
        "pairwise_F1": "{:.4f}",
        "pairwise_precision": "{:.4f}",
        "pairwise_recall": "{:.4f}",
        "ARI": "{:.4f}",
        "NMI": "{:.4f}",
    })
)


In [ ]:
# ==========================================================
# 5. Visual inspection of the best GT clustering (Optional)
# ==========================================================

print(
    f"Visualizing: {BEST_MODEL} | "
    f"{BEST_ALGORITHM} | {BEST_PARAMETER}"
)
print("Summary:", summarize_clusters(best_labels_eval))

# Set to True to display all members of the largest clusters.
SHOW_TOP_CLUSTERS = False

if SHOW_TOP_CLUSTERS:
    show_top_clusters(
        labels=best_labels_eval,
        paths=paths_eval,
        top_n=10,
        max_show=None,
        page_size=24,
        title=(
            f"Best GT configuration — {BEST_MODEL} — "
            f"{BEST_ALGORITHM} — {BEST_PARAMETER}"
        ),
    )

best_noise_indices = show_noise_for_model(
    labels=best_labels_eval,
    paths=paths_eval,
    model_name=(
        f"Best GT configuration — {BEST_MODEL} — "
        f"{BEST_ALGORITHM} — {BEST_PARAMETER}"
    ),
    max_show=80,
)

# Optional: inspect where predicted groups disagree with the GT.

def collect_pairwise_clustering_errors(
    labels,
    gold_groups,
):
    labels = np.asarray(labels)
    gold_groups = np.asarray(gold_groups)

    evaluation_indices = np.where(gold_groups != 0)[0]
    labels_local = labels[evaluation_indices]
    gold_local = gold_groups[evaluation_indices]

    upper_i, upper_j = np.triu_indices(len(evaluation_indices), k=1)

    true_same = gold_local[upper_i] == gold_local[upper_j]
    predicted_same = (
        (labels_local[upper_i] == labels_local[upper_j])
        & (labels_local[upper_i] != -1)
    )

    false_positive_mask = predicted_same & ~true_same
    false_negative_mask = ~predicted_same & true_same

    false_positive_pairs = [
        (
            int(evaluation_indices[i]),
            int(evaluation_indices[j]),
        )
        for i, j in zip(
            upper_i[false_positive_mask],
            upper_j[false_positive_mask],
        )
    ]

    false_negative_pairs = [
        (
            int(evaluation_indices[i]),
            int(evaluation_indices[j]),
        )
        for i, j in zip(
            upper_i[false_negative_mask],
            upper_j[false_negative_mask],
        )
    ]

    return false_positive_pairs, false_negative_pairs


false_positive_pairs, false_negative_pairs = (
    collect_pairwise_clustering_errors(
        labels=best_labels_eval,
        gold_groups=gold_eval,
    )
)

print("\nPairwise errors for the selected configuration:")
print("  False-positive pairs:", len(false_positive_pairs))
print("  False-negative pairs:", len(false_negative_pairs))
print("\nFirst 10 false-positive index pairs:")
print(false_positive_pairs[:10])
print("\nFirst 10 false-negative index pairs:")
print(false_negative_pairs[:10])


##Resources

- Lang, S.& Zinnen, M. (2025, February 26). Digital Provenance Research: Eine computerassistierte Bildersuche in historischen Auktionskatalogen. DHd 2025 Under Construction (DHd2025), Bielefeld, Deutschland. [DOI: https://doi.org/10.5281/zenodo.14943104]
- Smits, T., Warner, B., Fyfe, P., & Lee, B. C. G. (2025). A Fully Searchable Multimodal Dataset of the Illustrated London News, 1842–1890. Journal of Open Humanities Data, 11: 10, pp. 1–13. [DOI: https://doi.org/10.5334/johd.284]